In [4]:
import json
import os
import uuid
import urllib3
import requests
import pandas as pd
from sqlalchemy import create_engine, text

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = os.getenv("MES_BASE_URL", "https://localhost:7204")
SWAGGER_URL = f"{BASE_URL}/swagger/v1/swagger.json"

DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321",
}

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)


def fetch_swagger(url, headers=None, verify=False):
    response = requests.get(url, headers=headers or {"Accept": "application/json"}, verify=verify, timeout=20)
    response.raise_for_status()
    return response.json()


try:
    swagger = fetch_swagger(SWAGGER_URL)
    print("Swagger loaded successfully")
except Exception as e:
    print(f"Could not load swagger: {e}")
    swagger = {"paths": {}}


def parse_json_if_possible(value):
    if not value:
        return None
    if isinstance(value, dict) or isinstance(value, list):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value


rows = []
for path, methods in swagger.get("paths", {}).items():
    for method, details in methods.items():
        if method.lower() not in {"get", "post", "put", "patch", "delete"}:
            continue

        metadata = {}
        description = parse_json_if_possible(details.get("description"))
        if isinstance(description, dict):
            metadata.update(description)
        elif isinstance(description, str):
            metadata["description_text"] = description

        ext = details.get("x-metadata", {})
        if isinstance(ext, dict):
            metadata.update(ext)

        tool_name = (
            metadata.get("tool_name")
            or details.get("operationId")
            or f"{method.upper()}_{path.replace('/', '_')}"
        )

        rows.append({
            "tool_name": tool_name,
            "path": path,
            "method": method.upper(),
            "description": metadata.get("description") or details.get("summary") or "",
            "business_domain": metadata.get("business_domain") or "Unassigned",
            "recommended_agents": metadata.get("recommended_agents", []),
            "restricted_agents": metadata.get("restricted_agents", []),
            "agent_accessible": bool(metadata.get("agent_accessible", True)),
            "auto_register": bool(metadata.get("auto_register", True)),
            "operation_type": metadata.get("operation_type") or method.upper(),
            "risk_level": metadata.get("risk_level") or "MEDIUM",
            "priority": int(metadata.get("priority", 100)),
        })

backend_api_df = pd.DataFrame(rows)
print(f"Discovered APIs: {len(backend_api_df)}")
backend_api_df.head()

usable_api_df = backend_api_df[
    (backend_api_df["agent_accessible"] == True) &
    (backend_api_df["auto_register"] == True)
].copy()

usable_api_df = usable_api_df.sort_values(["business_domain", "priority", "tool_name"]).reset_index(drop=True)
print(f"Usable APIs for agentic pipelines: {len(usable_api_df)}")
usable_api_df[["tool_name", "business_domain", "operation_type", "priority"]].head(20)


def build_agent_definitions(api_df):
    agents = []
    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue
        agents.append({
            "agent_name": f"{domain} Agent",
            "business_domain": domain,
            "tool_count": len(group),
            "tools": group["tool_name"].tolist(),
            "registration_mode": "rule_based",
        })
    return pd.DataFrame(agents)


agent_registry_df = build_agent_definitions(usable_api_df)
agent_registry_df.head()


def build_rule_based_pipeline_registry(api_df):
    pipeline_rows = []
    pipeline_graph = {}

    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue

        ordered_group = group.sort_values(["priority", "tool_name"], ascending=[True, True]).reset_index(drop=True)
        selected_tools = ordered_group.head(5).copy()

        agent_name = f"{domain} Agent"
        pipeline_name = f"{domain.lower().replace(' ', '_')}_pipeline"

        graph_steps = []
        for idx, row in selected_tools.iterrows():
            step_name = f"step_{idx + 1}"
            step_id = f"{pipeline_name}_{step_name}"
            depends_on = [] if idx == 0 else [f"{pipeline_name}_step_{idx}"]
            graph_steps.append({
                "step_id": step_id,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "depends_on": depends_on,
            })

            pipeline_rows.append({
                "pipeline_name": pipeline_name,
                "agent_name": agent_name,
                "business_domain": domain,
                "step_order": idx + 1,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "path": row["path"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "uses_llm": False,
                "reason": "metadata_ordered_rule_based",
            })

        pipeline_graph[agent_name] = {
            "business_domain": domain,
            "pipeline_name": pipeline_name,
            "steps": graph_steps,
        }

    return pd.DataFrame(pipeline_rows), pipeline_graph


pipeline_registry_df, pipeline_graph = build_rule_based_pipeline_registry(usable_api_df)
pipeline_registry_df.head(15)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("PostgreSQL connected:", result.fetchone()[0])


def build_backend_api_rows(api_df):
    insert_rows = []
    for _, row in api_df.iterrows():
        metadata = {
            "tool_name": row.get("tool_name"),
            "business_domain": row.get("business_domain") or "Unassigned",
            "recommended_agents": row.get("recommended_agents") or [],
            "restricted_agents": row.get("restricted_agents") or [],
            "operation_type": row.get("operation_type") or "GET",
            "risk_level": row.get("risk_level") or "MEDIUM",
            "priority": int(row.get("priority", 100)),
            "agent_accessible": bool(row.get("agent_accessible", True)),
            "auto_register": bool(row.get("auto_register", True)),
        }
        insert_rows.append({
            "api_id": str(uuid.uuid4())[:50],
            "method": str(row["method"]).upper(),
            "end_point": row["path"],
            "description": row["description"] or "",
            "access_to_use": "allowed" if bool(row.get("agent_accessible", True)) else "restricted",
            "parameter": json.dumps(metadata),
            "error_status": json.dumps({"status": "discovered", "message": "ingested from swagger"}),
        })
    return insert_rows


insert_sql = """
INSERT INTO backend_api_table (
    api_id, method, end_point, description, access_to_use, parameter, error_status
)
VALUES (
    :api_id, :method, :end_point, :description, :access_to_use,
    CAST(:parameter AS jsonb),
    CAST(:error_status AS jsonb)
)
"""

try:
    insert_rows = build_backend_api_rows(backend_api_df)
    with engine.begin() as conn:
        for row in insert_rows:
            conn.execute(text(insert_sql), row)
    print(f"Inserted {len(insert_rows)} rows into backend_api_table")
except Exception as e:
    print(f"Could not write to backend_api_table: {e}")


Swagger loaded successfully
Discovered APIs: 670
Usable APIs for agentic pipelines: 362
PostgreSQL connected: PostgreSQL 16.14, compiled by Visual C++ build 1944, 64-bit
Inserted 670 rows into backend_api_table
